# Pretraining and Evaluation Notebook for NEPALI
This notebook implements: 
1. Dataset streaming from jsonl files.
2. Training loop with checkpoint resume and learning rate scheduling.
3. Validation tracking (cross-entropy, perplexity, and Bits-Per-Byte).
4. Autoregressive sample generation and metrics evaluation (BLEU-4, ROUGE-L, chrF, Distinct-1/2, repetition rate).
5. Attention analysis (heatmaps, entropy, and mean distance).
6. Automatic plotting and saving to `/figures` directory.

In [ ]:
import os
import sys
import json
import math
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tokenizers import Tokenizer
from collections import Counter
from torch.utils.data import IterableDataset, DataLoader

# Add project root to sys.path to enable package imports
sys.path.append(str(Path("../../")))
from transformer.arch import Transformer

In [ ]:
# Load model configurations
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)
print("Model Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

In [ ]:
class NepaliDataset(IterableDataset):
    """
    Dataset loader that streams tokenized sequences from jsonl files.
    Chops streams into contiguous blocks of max_seq_len plus 1 target token.
    """
    def __init__(self, file_path, max_seq_len):
        super().__init__()
        self.file_path = file_path
        self.max_seq_len = max_seq_len

    def __iter__(self):
        current_tokens = []
        with open(self.file_path, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)
                tokens = data.get("tokens", [])
                current_tokens.extend(tokens)
                
                while len(current_tokens) >= self.max_seq_len + 1:
                    segment = current_tokens[:self.max_seq_len + 1]
                    current_tokens = current_tokens[self.max_seq_len:]
                    yield torch.tensor(segment[:-1]), torch.tensor(segment[1:])


In [ ]:
def save_checkpoint(model, optimizer, scheduler, step, train_losses, val_losses, filename='checkpoint.pth', checkpoint_dir='../checkpoints'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'step': step,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'config': config
    }
    torch.save(checkpoint, os.path.join(checkpoint_dir, filename))
    print(f"[INFO] Checkpoint saved successfully: {filename} at step {step}.")

def load_checkpoint(model, optimizer, scheduler, checkpoint_path="../checkpoints/checkpoint.pth"):
    if os.path.exists(checkpoint_path):
        print(f"[INFO] Resuming training from existing checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=next(model.parameters()).device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        return checkpoint['step'], checkpoint.get('train_losses', []), checkpoint.get('val_losses', [])
    print("[INFO] No checkpoint found. Initializing model training from scratch.")
    return 0, [], []

In [ ]:
# Setup device GPU/MPS acceleration
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using device: {device}")

# Initialize model and optimizer
model = Transformer(**config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=6e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100000, eta_min=6e-5)

# Load checkpoint
start_step, train_losses, val_losses = load_checkpoint(model, optimizer, scheduler)

# Dataset paths
train_path = f"../datasets/cleaned/{lang}_train.jsonl"
val_path = f"../datasets/cleaned/{lang}_val.jsonl"

print(f"Pretraining {lang.upper()} model...")

def run_pretraining(max_steps=5000, batch_size=32, grad_accum_steps=4):
    step = start_step
    if step >= max_steps:
        print("[INFO] Already pretrained to target steps.")
        return
        
    dataset = NepaliDataset(train_path, config['max_seq_len'])
    loader = DataLoader(dataset, batch_size=batch_size)
    
    best_val_loss = min(val_losses) if len(val_losses) > 0 else float('inf')
    running_loss = 0.0
    optimizer.zero_grad()
    
    for x, y in loader:
        if step >= max_steps:
            break
            
        model.train()
        x, y = x.to(device), y.to(device)
        
        logits, loss, _ = model(x, y)
        # Normalize loss for gradient accumulation
        loss = loss / grad_accum_steps
        loss.backward()
        running_loss += loss.item() * grad_accum_steps
        
        if (step + 1) % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            train_losses.append(running_loss / grad_accum_steps)
            running_loss = 0.0
            
            # Validation cycle
            if step % 500 == 0:
                if 'evaluate_loss' in globals():
                    val_loss, ppl, bpb = evaluate_loss(model, val_path, batch_size)
                    val_losses.append(val_loss)
                    print(f"Step {step} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_loss:.4f} | PPL: {ppl:.2f} | BPB: {bpb:.4f}")
                    
                    # Update best checkpoint if validation loss improves
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        save_checkpoint(model, optimizer, scheduler, step, train_losses, val_losses, filename='best_checkpoint.pth')
                        print(f"New best validation loss achieved: {best_val_loss:.4f}. Saved best_checkpoint.pth")
                else:
                    print(f"Step {step} | Train Loss: {train_losses[-1]:.4f} | [Validation skipped: evaluate_loss function not defined yet]")
                
            # Save checkpoints (latest state for resume)
            if step % 1000 == 0:
                save_checkpoint(model, optimizer, scheduler, step, train_losses, val_losses, filename='checkpoint.pth')
                
        step += 1
        
# Run training (commented to prevent auto-run in notebook)
# run_pretraining(max_steps=5000)

## 2. Loss Curve Visualization

Plots the training and validation cross-entropy losses over training steps to diagnose optimization progress, convergence speed, and potential overfitting.

In [ ]:
def save_loss_curve(train_losses, val_losses, figures_dir="../figures"):
    """Generates and saves the loss training curve."""
    os.makedirs(figures_dir, exist_ok=True)
    plt.figure(figsize=(9, 5))
    plt.plot(train_losses, label='Train Loss', color='royalblue')
    # Stretch val losses to match train loss step logs if evaluated at intervals
    if len(val_losses) > 0:
        val_steps = np.linspace(0, len(train_losses), len(val_losses))
        plt.plot(val_steps, val_losses, label='Validation Loss', color='darkorange', marker='o')
    plt.xlabel('Training Steps')
    plt.ylabel('Cross-Entropy Loss')
    plt.title(f'Pretraining Loss Curve ({lang.upper()})')
    plt.legend()
    plt.grid(True, linestyle='--')
    plot_path = os.path.join(figures_dir, 'loss_curve.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Loss curve saved to {plot_path}")

## 3. Intrinsic Language Modeling Metrics

To evaluate the predictive quality of our language model, we compute three standard intrinsic metrics:

### A. Cross-Entropy Loss
Measures the performance of the classification model whose output is a probability value between 0 and 1:
$$L = -\frac{1}{T} \sum_{t=1}^T \log P(x_t \mid x_{<t})$$

### B. Perplexity (PPL)
Represents the exponentiated cross-entropy loss, representing the average branching factor of the model's vocabulary at each token prediction:
$$\text{PPL} = e^L$$

### C. Bits-Per-Byte (BPB)
Calculates the average number of bits required to encode each raw text character. This is critical for comparing models across languages with different tokenizers and character fertility rates:
$$\text{BPB} = \frac{L \cdot N_{\text{tokens}}}{N_{\text{chars}} \cdot \ln(2)}$$

In [ ]:
def calculate_bpb(loss_val, num_tokens, num_chars):
    """
    Computes Bits-Per-Byte (BPB) metric comparing token losses on a character scale.
    Formula: (Loss * num_tokens) / (num_chars * ln(2))
    """
    return (loss_val * num_tokens) / (num_chars * math.log(2))

In [ ]:
def evaluate_loss(model, dataset_path, batch_size=32):
    """Runs validation evaluation, returning average loss, perplexity, and bits-per-byte."""
    model.eval()
    dataset = NepaliDataset(dataset_path, config['max_seq_len'])
    loader = DataLoader(dataset, batch_size=batch_size)
    
    total_loss = 0.0
    num_batches = 0
    num_tokens = 0
    num_chars = 0
    
    # We need character length of validation file to compute exact BPB
    # We sum characters in the raw lines of the dataset
    with open(dataset_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            num_chars += len(data.get("text", ""))
            
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(next(model.parameters()).device), y.to(next(model.parameters()).device)
            _, loss, _ = model(x, y)
            total_loss += loss.item()
            num_batches += 1
            num_tokens += y.numel()
            
    if num_batches == 0:
        return float('inf'), float('inf'), float('inf')
        
    avg_loss = total_loss / num_batches
    perplexity = math.exp(avg_loss)
    bpb = calculate_bpb(avg_loss, num_tokens, num_chars)
    
    return avg_loss, perplexity, bpb

## 3. Text Generation

This function implements autoregressive next-token prediction, supporting greedy decoding (temperature = 0) and temperature-scaled stochastic sampling.

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=40, temperature=1.0):
    """Generates text continuation from a prompt using greedy or temperature sampling."""
    model.eval()
    tokens = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(tokens).unsqueeze(0).to(device)
    
    for _ in range(max_new_tokens):
        cond_input = input_ids[:, -config['max_seq_len']:]
        with torch.no_grad():
            logits, _, _ = model(cond_input)
        
        next_logits = logits[:, -1, :]
        
        if temperature == 0.0:
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
        else:
            next_logits = next_logits / temperature
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
        input_ids = torch.cat((input_ids, next_token), dim=-1)
        
    return tokenizer.decode(input_ids[0].tolist())

## 4. BLEU-4 (Bilingual Evaluation Understudy)

BLEU computes n-gram precision (from 1-grams to 4-grams) between the candidate sentence and the reference text. A brevity penalty (BP) is applied to prevent short candidates from scoring artificially high:
$$\text{BLEU-4} = \text{BP} \cdot \exp\left( \sum_{n=1}^4 0.25 \cdot \ln(p_n) \right)$$
Where $p_n$ is the smoothed precision for n-grams.

In [ ]:
def compute_bleu_4(reference_text, candidate_text):
    """Computes Sentence BLEU-4 score from scratch."""
    ref_words = reference_text.split()
    cand_words = candidate_text.split()
    if not ref_words or not cand_words:
        return 0.0
        
    p_ns = []
    for n in range(1, 5):
        ref_g = [tuple(ref_words[i:i+n]) for i in range(len(ref_words) - n + 1)]
        cand_g = [tuple(cand_words[i:i+n]) for i in range(len(cand_words) - n + 1)]
        if not ref_g or not cand_g:
            p_ns.append(1e-9)
            continue
        ref_c, cand_c = Counter(ref_g), Counter(cand_g)
        shared = sum(min(cand_c[g], ref_c[g]) for g in cand_c)
        p_ns.append(shared / len(cand_g) if len(cand_g) > 0 else 1e-9)
        
    p_ns = [p if p > 0 else 1e-9 for p in p_ns]
    c_len = len(cand_words)
    r_len = len(ref_words)
    bp = 1.0 if c_len > r_len else (math.exp(1 - r_len / c_len) if c_len > 0 else 0.0)
    return bp * math.exp(sum(0.25 * math.log(p) for p in p_ns))

## 5. chrF (Character n-gram F-score)

chrF computes character-level n-gram precision ($P$) and recall ($R$) (typically up to 6-grams), combined using a $\beta$ parameter to bias toward recall (standard $\beta=3.0$):
$$\text{chrF} = (1 + \beta^2) \frac{P \cdot R}{\beta^2 P + R}$$

In [ ]:
def compute_chrf(reference_text, candidate_text):
    """Computes character n-gram F-score (chrF) with beta=3.0."""
    def get_char_ngrams(text, n):
        clean = text.replace(" ", "")
        return [clean[i:i+n] for i in range(len(clean) - n + 1)]
        
    precisions, recalls = [], []
    for i in range(1, 7):
        ref_g = get_char_ngrams(reference_text, i)
        cand_g = get_char_ngrams(candidate_text, i)
        if not ref_g or not cand_g:
            continue
        ref_c, cand_c = Counter(ref_g), Counter(cand_g)
        shared = sum(min(cand_c[g], ref_c[g]) for g in cand_c)
        precisions.append(shared / len(cand_g))
        recalls.append(shared / len(ref_g))
        
    avg_p = sum(precisions) / len(precisions) if precisions else 0.0
    avg_r = sum(recalls) / len(recalls) if recalls else 0.0
    if (avg_p + avg_r) == 0:
        return 0.0
    return (10.0 * avg_p * avg_r) / (9.0 * avg_p + avg_r)

## 6. ROUGE-L (Recall-Oriented Understudy for Gisting Evaluation - LCS)

ROUGE-L computes the longest common subsequence (LCS) ratio between candidate and reference words to evaluate sequence structure retention:
$$\text{ROUGE-L} = \frac{2 \cdot R_{\text{LCS}} \cdot P_{\text{LCS}}}{R_{\text{LCS}} + P_{\text{LCS}}}$$

In [ ]:
def compute_rouge_l(reference_text, candidate_text):
    """Computes sentence ROUGE-L score based on Longest Common Subsequence."""
    ref_words = reference_text.split()
    cand_words = candidate_text.split()
    m, n = len(ref_words), len(cand_words)
    if m == 0 or n == 0:
        return 0.0
        
    L = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 or j == 0:
                L[i][j] = 0
            elif ref_words[i-1] == cand_words[j-1]:
                L[i][j] = L[i-1][j-1] + 1
            else:
                L[i][j] = max(L[i-1][j], L[i][j-1])
                
    lcs_len = L[m][n]
    r_lcs = lcs_len / m
    p_lcs = lcs_len / n
    if (r_lcs + p_lcs) == 0:
        return 0.0
    return (2 * r_lcs * p_lcs) / (r_lcs + p_lcs)

## 7. Distinct-1 / Distinct-2

Distinct-N measures vocabulary diversity by calculating the ratio of unique n-grams over the total generated tokens.

In [ ]:
def compute_distinct_n(tokens, n=1):
    """Distinct-N metric (fraction of unique n-grams)."""
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    return len(set(ngrams)) / len(ngrams)

## 8. Repetition Rate

Measures the fraction of repeating 4-grams in generated sequences to identify loop collapse during text continuation.

In [ ]:
def compute_repetition_rate(tokens, n=4):
    """Repetition rate diagnostic (fraction of repeating n-grams)."""
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    repeated = len(ngrams) - len(set(ngrams))
    return repeated / len(ngrams)

## 9. Attention Matrix Analysis

To understand what the model attends to inside its transformer layers, we extract the query-key attention matrix $A_{h, i, j}$ for Layer $L$, Head $H$, and compute the following head diagnostics:

### A. Shannon Attention Entropy
Measures the dispersion of attention weight per token. High entropy indicates diffuse attention across all tokens; low entropy indicates sharp focus on a few specific tokens:
$$H(A_h) = -\frac{1}{T} \sum_{i=1}^T \sum_{j=1}^T A_{h, i, j} \log_2(A_{h, i, j} + \epsilon)$$

### B. Mean Attention Distance
Quantifies the physical span of attention. Short distance indicates local/positional attention (attending to nearby tokens); long distance indicates global/content attention (attending to far-away context):
$$D(A_h) = \frac{1}{T} \sum_{i=1}^T \sum_{j=1}^T A_{h, i, j} \cdot |i - j|$$

In [ ]:
def plot_attention_heatmap(attn_matrix, layer_idx, head_idx, tokens, figures_dir="../figures"):
    """Generates query-key attention heatmaps and saves them to the figures directory."""
    os.makedirs(figures_dir, exist_ok=True)
    T = len(tokens)
    matrix = attn_matrix[0, head_idx, :T, :T].cpu().numpy()
    
    plt.figure(figsize=(8, 7))
    sns.heatmap(
        matrix, xticklabels=tokens, yticklabels=tokens, 
        cmap="viridis", square=True, cbar_kws={'label': 'Attention Weight'}
    )
    plt.title(f"Attention Heatmap: Layer {layer_idx}, Head {head_idx} ({lang.upper()})")
    plt.xlabel("Key Positions (Source)")
    plt.ylabel("Query Positions (Target)")
    
    plot_path = os.path.join(figures_dir, f'attention_layer{layer_idx}_head{head_idx}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Attention heatmap saved successfully at {plot_path}")

def compute_attention_metrics(attn_matrix):
    """
    Computes Shannon Entropy and Mean Attention Distance per head.
    attn_matrix shape: (1, n_heads, T, T)
    """
    _, n_heads, T, _ = attn_matrix.size()
    entropies = []
    distances = []
    
    for h in range(n_heads):
        head_matrix = attn_matrix[0, h, :, :] # (T, T)
        
        # 1. Shannon Entropy: -sum(p * log(p))
        entropy = -torch.sum(head_matrix * torch.log(head_matrix + 1e-9), dim=-1).mean().item()
        entropies.append(entropy)
        
        # 2. Mean Attention Distance: sum(p * |i - j|)
        indices = torch.arange(T, dtype=torch.float32, device=attn_matrix.device)
        dist_matrix = torch.abs(indices.unsqueeze(0) - indices.unsqueeze(1)) # (T, T)
        mean_dist = torch.sum(head_matrix * dist_matrix, dim=-1).mean().item()
        distances.append(mean_dist)
        
    return entropies, distances

## 10. Performance Evaluation Demo Run

Loads the tokenizer, runs evaluation on the validation dataset (PPL and BPB), generates text continuation from a prompt, computes fluency/diversity metrics, and outputs attention heatmap plots.

In [ ]:
# Load tokenizer
tokenizer = Tokenizer.from_file(f"../tokenizer/{lang}_tokenizer.json")

print(f"=== Performance Evaluation Demo for {lang.upper()} ===")

# 1. Run intrinsic evaluations on validation data
try:
    val_loss, ppl, bpb = evaluate_loss(model, val_path)
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Perplexity (PPL): {ppl:.2f}")
    print(f"Bits-Per-Byte (BPB): {bpb:.4f}")
except Exception as e:
    print(f"[WARNING] Validation evaluation failed: {e}")

# 2. Run autoregressive text generation and fluency diagnostics
prompt = "नेपाल एक सुन्दर देश हो"
try:
    gen_out = generate_text(model, tokenizer, prompt, max_new_tokens=40, temperature=0.7)
    print(f"\nPrompt: {prompt}")
    print(f"Generated Continuation: {gen_out}")
    
    gen_tokens = tokenizer.encode(gen_out).ids
    print(f"\nFluency & Diversity Diagnostics:")
    print(f"  Distinct-1: {compute_distinct_n(gen_tokens, 1):.4f}")
    print(f"  Distinct-2: {compute_distinct_n(gen_tokens, 2):.4f}")
    print(f"  Repetition Rate (4-grams): {compute_repetition_rate(gen_tokens, 4):.4f}")
except Exception as e:
    print(f"[WARNING] Text generation failed: {e}")

# 3. Run attention visualization on the generated sequence
try:
    input_ids = torch.tensor(tokenizer.encode(prompt).ids).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        _, _, all_attns = model(input_ids)
    
    # Extract tokens for plotting labels
    tokens = [tokenizer.decode([tid]) for tid in tokenizer.encode(prompt).ids]
    
    # Plot Layer 0, Head 0
    plot_attention_heatmap(all_attns[0], layer_idx=0, head_idx=0, tokens=tokens)
    
    # Plot Layer 5 (late), Head 0
    plot_attention_heatmap(all_attns[-1], layer_idx=len(all_attns)-1, head_idx=0, tokens=tokens)
    
    # Calculate attention metrics
    entropies, distances = compute_attention_metrics(all_attns[-1])
    print(f"\nAttention Head Statistics (Layer {len(all_attns)-1}):")
    for h in range(len(entropies)):
        print(f"  Head {h} | Shannon Entropy: {entropies[h]:.4f} | Mean Attention Distance: {distances[h]:.2f}")
except Exception as e:
    print(f"[WARNING] Attention visualization failed: {e}")